<a href="https://colab.research.google.com/github/HanaaMaswada/RTT-Correlation-with-Transfer-phases-of-a-file-download/blob/main/impact_of_network_round_trip_time_on_tcp_connection_establishment_and_time_to_first_byte_during_http_file_downloads_an_empirical_measurement_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1 — Upload the Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

Step 2 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Step 3 — Read the CSV File

In [ ]:
df = pd.read_csv("phase1_results (1).csv")

Step 4 — Display the First Rows

In [ ]:
df.head()

Step 5 — Check Dataset Shape

In [ ]:
df.shape

Step 6 — Display Column Names

In [ ]:
df.columns

Step 7 — Display Dataset Information

In [ ]:
df.info()

Step 8 — Keep Only Successful Downloads

In [ ]:
df = df[df["curl_status"] == "ok"]

Step 9 — Keep Only Successful Ping Measurements

In [ ]:
df = df[df["ping_status"] == "ok"]

Step 10 — Keep Only 100MB Downloads

In [ ]:
df = df[df["size_download_bytes"] == 104857600]

Step 11 — Check Dataset After Cleaning

In [ ]:
df.shape

Step 12 — Select Important Columns

In [ ]:
important_cols = [
    "server_name",
    "rtt_avg_ms",
    "dns_time_s",
    "tcp_connect_time_s",
    "tls_handshake_time_s",
    "ttfb_s",
    "throughput_Mbps",
    "total_time_s"
]

df = df[important_cols]

Step 13 — Display Cleaned Dataset

In [ ]:
df.head()

Step 14 — Descriptive Statistics

In [ ]:
df.describe()

Step 15 — Set Visualization Style

In [ ]:
sns.set(style="whitegrid")

Step 16 — Plot RTT Distribution

In [ ]:
plt.figure(figsize=(8,6))

sns.histplot(df["rtt_avg_ms"], bins=20)

plt.title("RTT Distribution")
plt.xlabel("RTT (ms)")
plt.ylabel("Count")
sns.histplot(kde=True)

plt.show()

Step 17 — Plot RTT vs TTFB

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x="rtt_avg_ms",
    y="ttfb_s",
    scatter_kws={"alpha":0.6}
)

plt.title("RTT vs TTFB")
plt.xlabel("RTT (ms)")
plt.ylabel("TTFB (s)")

plt.show()

Step 18 — Plot RTT vs Throughput

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x="rtt_avg_ms",
    y="throughput_Mbps",
    scatter_kws={"alpha":0.6}
)

plt.title("RTT vs Throughput")
plt.xlabel("RTT (ms)")
plt.ylabel("Throughput (Mbps)")

plt.show()

Step 19 — Plot RTT vs TCP Connection Time

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x="rtt_avg_ms",
    y="tcp_connect_time_s"
)

plt.title("RTT vs TCP Connection Time")
plt.xlabel("RTT (ms)")
plt.ylabel("TCP Connection Time (s)")

plt.show()

Step 20 — Plot Throughput Distribution per Server

In [ ]:
plt.figure(figsize=(14,6))

sns.boxplot(
    data=df,
    x="server_name",
    y="throughput_Mbps"
)

plt.xticks(rotation=45)

plt.title("Throughput Distribution per Server")
plt.xlabel("Server")
plt.ylabel("Throughput (Mbps)")

plt.show()

Step 21 — Plot Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))

corr = df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

Step 22 — Import Pearson Correlation Function

In [ ]:
from scipy.stats import pearsonr

Step 23 — Calculate Pearson Correlation Between RTT and TTFB

In [ ]:
pearsonr(df["rtt_avg_ms"], df["ttfb_s"])

Step 24 — Calculate Pearson Correlation Between RTT and Throughput

In [ ]:
pearsonr(df["rtt_avg_ms"], df["throughput_Mbps"])

Step 25 — Calculate Pearson Correlation Between RTT and TCP Connection Time

In [ ]:
pearsonr(df["rtt_avg_ms"], df["tcp_connect_time_s"])

##Pairplot

In [ ]:
sns.pairplot(df)

Step 26 — Create Server Delay Column

In [ ]:
df["server_delay_s"] = (
    df["ttfb_s"]
    - df["dns_time_s"]
    - df["tcp_connect_time_s"]
    - df["tls_handshake_time_s"]
)

Step 27 — Calculate Average Phase Times per Server

In [ ]:
phase_avg = df.groupby("server_name")[[
    "dns_time_s",
    "tcp_connect_time_s",
    "tls_handshake_time_s",
    "server_delay_s"
]].mean()

Step 28 — Display Phase Breakdown Table

In [ ]:
phase_avg.head()

Step 29 — Plot Phase Breakdown Chart

In [ ]:
server_rtt = df.groupby("server_name")["rtt_avg_ms"].mean()

phase_avg["avg_rtt"] = server_rtt

phase_avg = phase_avg.sort_values("avg_rtt")

In [ ]:
phase_avg.plot(
    kind="bar",
    stacked=True,
    figsize=(14,7)
)

plt.title("Average TTFB Breakdown per Server")
plt.xlabel("Server")
plt.ylabel("Time (seconds)")

plt.legend(title="Download Phase")

plt.xticks(rotation=45)

plt.show()

Step 30 — Plot RTT vs TLS Handshake Time

RTT vs TLS Handshake Time

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x="rtt_avg_ms",
    y="tls_handshake_time_s",
    scatter_kws={"alpha":0.6}
)

plt.title("RTT vs TLS Handshake Time")
plt.xlabel("RTT (ms)")
plt.ylabel("TLS Handshake Time (s)")

plt.show()

Step 31 — Plot RTT vs Total Download Time

In [ ]:
plt.figure(figsize=(8,6))

sns.regplot(
    data=df,
    x="rtt_avg_ms",
    y="total_time_s",
    scatter_kws={"alpha":0.6}
)

plt.title("RTT vs Total Download Time")
plt.xlabel("RTT (ms)")
plt.ylabel("Total Download Time (s)")

plt.show()

Step 32 — Plot Average RTT per Server

In [ ]:
avg_rtt = df.groupby("server_name")["rtt_avg_ms"].mean()

plt.figure(figsize=(14,6))

avg_rtt.sort_values().plot(kind="bar")

plt.title("Average RTT per Server")
plt.xlabel("Server")
plt.ylabel("Average RTT (ms)")

plt.xticks(rotation=45)

plt.show()

tep 33 — Plot Average Throughput per Server

In [ ]:
avg_thr = df.groupby("server_name")["throughput_Mbps"].mean()

plt.figure(figsize=(14,6))

avg_thr.sort_values().plot(kind="bar")

plt.title("Average Throughput per Server")
plt.xlabel("Server")
plt.ylabel("Average Throughput (Mbps)")

plt.xticks(rotation=45)

plt.show()

Step 34 — Plot RTT vs Throughput by Server

In [ ]:
plt.figure(figsize=(10,7))

sns.scatterplot(
    data=df,
    x="rtt_avg_ms",
    y="throughput_Mbps",
    hue="server_name"
)

plt.title("RTT vs Throughput by Server")
plt.xlabel("RTT (ms)")
plt.ylabel("Throughput (Mbps)")

plt.show()


Step 35 — Plot Pairplot

In [ ]:
sns.pairplot(
    df[[
        "rtt_avg_ms",
        "tcp_connect_time_s",
        "tls_handshake_time_s",
        "ttfb_s",
        "throughput_Mbps"
    ]]
)

plt.show()

Step 36 — Plot RTT KDE Distribution

In [ ]:
plt.figure(figsize=(8,6))

sns.kdeplot(df["rtt_avg_ms"], fill=True)

plt.title("RTT Density Distribution")
plt.xlabel("RTT (ms)")

plt.show()

Step 37 — Plot TTFB Distribution per Server

In [ ]:
plt.figure(figsize=(14,6))

sns.boxplot(
    data=df,
    x="server_name",
    y="ttfb_s"
)

plt.xticks(rotation=45)

plt.title("TTFB Distribution per Server")
plt.xlabel("Server")
plt.ylabel("TTFB (s)")

plt.show()

In [ ]:
from scipy.stats import pearsonr

pearsonr(df["rtt_avg_ms"], df["tcp_connect_time_s"])

In [ ]:
pearsonr(df["rtt_avg_ms"], df["ttfb_s"])

In [ ]:
pearsonr(df["rtt_avg_ms"], df["total_time_s"])

In [ ]:
df.shape

اضافي

In [ ]:
import pandas as pd

df = pd.read_csv("phase1_results (1).csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df["server_name"].value_counts()

In [ ]:
df_clean = df[
    (df["curl_status"] == "ok") &
    (df["ping_status"] == "ok") &
    (df["size_download_bytes"] == 104857600)
]

df_clean["server_name"].value_counts()

In [ ]:
print("Valid observations:", len(df_clean))
print("Unique servers:", df_clean["server_name"].nunique())

In [ ]:
[col for col in df.columns if "trial" in col.lower() or "run" in col.lower()]

In [ ]:
df_clean.groupby("server_name")["trial"].nunique()